## Approach 1: Standard One-Hot Encoding & Imputation

- Handled missing values using **median imputation**.
- Applied **One-Hot Encoding** to categorical variables.
- Resulted in **68 total features** after encoding.

---

## Approach 2: Feature Engineering (Polynomial Interactions)

- Generated new interaction terms between existing features using **PolynomialFeatures**.
- Expanded the dataset to **over 1,700 features**.
- Designed to capture complex, non-linear relationships.

---

## Approach 3: Label Encoding & Custom Cleaning

- Applied domain-specific logic to clean missing data  
  - Example: inferring sex from relationship status.
- Used **SimpleImputer**:
  - Mean for numerical features  
  - Most frequent for categorical features
- Applied **Label Encoding** to categorical variables instead of One-Hot Encoding.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import  confusion_matrix
from itertools import combinations
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
data1=pd.read_csv('/content/data.csv', na_values=['#NAME?'])
data1.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39.0,State-gov,77516.0,Bachelors,13.0,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50.0,Self-emp-not-inc,83311.0,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38.0,Private,215646.0,HS-grad,9.0,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53.0,Private,234721.0,11th,7.0,Married-civ-spouse,Handlers-cleaners,Husband,Black,NaN,0,0,40,United-States,<=50K
4,28.0,Private,338409.0,Bachelors,13.0,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [ ]:
data1.shape

(5000, 15)

In [ ]:
data1['income'] = [0 if x == '<=50K' else 1 for x in data1['income']]

In [ ]:
X = data1.drop('income', axis='columns')
Y = data1['income']

In [ ]:
X.isnull().sum()

,0
age,48
workclass,0
fnlwgt,107
education,0
education_num,57
marital_status,0
occupation,0
relationship,0
race,264
sex,47


In [ ]:
for col_name in X.columns:
    if X[col_name].dtypes == 'object':
        unique_cat = len(X[col_name].unique())
        print(f"Feature '{col_name}' has {unique_cat} unique categories")

Feature 'workclass' has 8 unique categories
Feature 'education' has 17 unique categories
Feature 'marital_status' has 7 unique categories
Feature 'occupation' has 15 unique categories
Feature 'relationship' has 6 unique categories
Feature 'race' has 6 unique categories
Feature 'sex' has 3 unique categories
Feature 'native_country' has 40 unique categories


In [ ]:
X['native_country'] = ['United-States ' if x == 'United-States' else 'Other' for x in X['native_country']]

In [ ]:
todummy_list = ['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country']

In [ ]:
def dummy_df(df, todummy_list):
    for x in todummy_list:
        dummies = pd.get_dummies(df[x], prefix=x)
        df = df.drop(x,  axis=1)
        df = pd.concat([df, dummies], axis=1)
    return df

In [ ]:
X = dummy_df(X, todummy_list)
X.shape

(5000, 68)

In [ ]:
imputer = SimpleImputer(missing_values=np.nan, strategy='median')

imputer.fit(X)
X = pd.DataFrame(data=imputer.transform(X) , columns=X.columns)

In [ ]:
X.isnull().sum()

,0
age,0
fnlwgt,0
education_num,0
capital_gain,0
capital_loss,0
...,...
race_White,0
sex_Female,0
sex_Male,0
native_country_Other,0


In [ ]:
X.head()

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week,workclass_?,workclass_Federal-gov,workclass_Local-gov,workclass_Private,...,relationship_Wife,race_Amer-Indian-Eskimo,race_Asian-Pac-Islander,race_Black,race_Other,race_White,sex_Female,sex_Male,native_country_Other,native_country_United-States
0,39.0,77516.0,13.0,2174.0,0.0,40.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
1,50.0,83311.0,13.0,0.0,0.0,13.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
2,38.0,215646.0,9.0,0.0,0.0,40.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
3,53.0,234721.0,7.0,0.0,0.0,40.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
4,28.0,338409.0,13.0,0.0,0.0,40.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0


## Applying logistic

In [ ]:
xtrain,xtest,ytrain,ytest= train_test_split(X,Y,test_size=0.3,random_state=1)

In [ ]:

logmodel = LogisticRegression()
logmodel.fit(xtrain,ytrain)

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [ ]:
logmodel.score(xtest,ytest)

0.8066666666666666

In [ ]:
predictions = logmodel.predict(xtest)
predictions

array([0, 0, 0, ..., 0, 0, 0])

In [ ]:
ytest.shape

(1500,)

In [ ]:
print(classification_report(ytest,predictions))

              precision    recall  f1-score   support

           0       0.82      0.97      0.88      1149
           1       0.72      0.28      0.41       351

    accuracy                           0.81      1500
   macro avg       0.77      0.63      0.65      1500
weighted avg       0.79      0.81      0.77      1500



In [ ]:
print(confusion_matrix(ytest, predictions))

[[1110   39]
 [ 251  100]]


In [ ]:
def add_interactions(dat):

    combos = list(combinations(list(dat.columns), 2))
    colnames = list(dat.columns) + ['_'.join(x) for x in combos]


    poly = PolynomialFeatures(interaction_only=True, include_bias=False)
    dat = poly.fit_transform(dat)
    dat = pd.DataFrame(dat)
    dat.columns = colnames


    noint_indicies = [i for i, x in enumerate(list((dat == 0).all())) if x]
    dat = dat.drop(dat.columns[noint_indicies], axis=1)

    return dat

In [ ]:
X1=add_interactions(X)

In [ ]:
X1.shape

(5000, 1737)

applying logistic

In [ ]:
x1train,x1test,ytrain,ytest= train_test_split(X1,Y,test_size=0.3,random_state=1)

In [ ]:
logmodel = LogisticRegression()
logmodel.fit(x1train,ytrain)

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [ ]:
logmodel.score(x1test,ytest)

0.8326666666666667

In [ ]:
predictions = logmodel.predict(x1test)
predictions

array([0, 0, 0, ..., 0, 0, 0])

In [ ]:
print(classification_report(ytest,predictions))

              precision    recall  f1-score   support

           0       0.87      0.92      0.89      1149
           1       0.67      0.55      0.61       351

    accuracy                           0.83      1500
   macro avg       0.77      0.74      0.75      1500
weighted avg       0.82      0.83      0.83      1500



In [ ]:
print(confusion_matrix(ytest, predictions))

[[1055   94]
 [ 157  194]]


In [ ]:
data3=data1

In [ ]:
data3.nunique()

,0
age,68
workclass,8
fnlwgt,4506
education,17
education_num,16
marital_status,7
occupation,15
relationship,6
race,5
sex,2


In [ ]:
data3['native_country'] = ['United-States ' if x == 'United-States' else 'Other' for x in data3['native_country']]
data3

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39.0,State-gov,77516.0,Bachelors,13.0,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50.0,Self-emp-not-inc,83311.0,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38.0,Private,215646.0,HS-grad,9.0,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53.0,Private,234721.0,11th,7.0,Married-civ-spouse,Handlers-cleaners,Husband,Black,NaN,0,0,40,United-States,<=50K
4,28.0,Private,338409.0,Bachelors,13.0,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Other,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,43.0,Private,222971.0,5th-6th,3.0,Never-married,Machine-op-inspct,Unmarried,White,Female,0,0,40,Other,<=50K
4996,31.0,Private,259425.0,HS-grad,9.0,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,>50K
4997,47.0,Self-emp-inc,212120.0,HS-grad,9.0,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,>50K
4998,NaN,Private,245880.0,HS-grad,9.0,Never-married,Adm-clerical,Not-in-family,White,Male,0,0,60,United-States,<=50K


In [ ]:
data3['fnlwgt'].median()

179533.0

In [ ]:
data3['fnlwgt'] = [0 if x < 179533 else 1 for x in data3['fnlwgt']]

In [ ]:
data3

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39.0,State-gov,0,Bachelors,13.0,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50.0,Self-emp-not-inc,0,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38.0,Private,1,HS-grad,9.0,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53.0,Private,1,11th,7.0,Married-civ-spouse,Handlers-cleaners,Husband,Black,NaN,0,0,40,United-States,<=50K
4,28.0,Private,1,Bachelors,13.0,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Other,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,43.0,Private,1,5th-6th,3.0,Never-married,Machine-op-inspct,Unmarried,White,Female,0,0,40,Other,<=50K
4996,31.0,Private,1,HS-grad,9.0,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,>50K
4997,47.0,Self-emp-inc,1,HS-grad,9.0,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,>50K
4998,NaN,Private,1,HS-grad,9.0,Never-married,Adm-clerical,Not-in-family,White,Male,0,0,60,United-States,<=50K


In [ ]:
data3.loc[data3['relationship']=='Husband','sex']='Male'

In [ ]:
data3.loc[data3['relationship']=='Wife','sex']='Female'

In [ ]:
data3.isnull().sum()

,0
age,48
workclass,0
fnlwgt,0
education,0
education_num,57
marital_status,0
occupation,0
relationship,0
race,264
sex,23


In [ ]:
data3['sex']=data3['sex'].fillna(data3['sex'].mode())

In [ ]:
data3['race']=data3['race'].fillna('Other')

In [ ]:
data3['age']=data3['age'].fillna(data3['age'].mean())

In [ ]:
data3['education_num']=data3['education_num'].fillna(data3['education_num'].mean())

In [ ]:
data3['race'].value_counts()

,count
race,
White,4021
Black,493
Other,293
Asian-Pac-Islander,145
Amer-Indian-Eskimo,48


In [ ]:
data3.isnull().sum()

,0
age,0
workclass,0
fnlwgt,0
education,0
education_num,0
marital_status,0
occupation,0
relationship,0
race,0
sex,23


In [ ]:
simplemode=SimpleImputer(strategy='most_frequent')

In [ ]:
data3[['sex']]=simplemode.fit_transform(data3[['sex']])

In [ ]:
data3.isnull().sum()

,0
age,0
workclass,0
fnlwgt,0
education,0
education_num,0
marital_status,0
occupation,0
relationship,0
race,0
sex,0


In [ ]:
data3.nunique()

,0
age,69
workclass,8
fnlwgt,2
education,17
education_num,17
marital_status,7
occupation,15
relationship,6
race,5
sex,2


In [ ]:
data3['income'] = [0 if x == '<=50K' else 1 for x in data3['income']]

In [ ]:
data3

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39.00000,State-gov,0,Bachelors,13.0,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,0
1,50.00000,Self-emp-not-inc,0,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,0
2,38.00000,Private,1,HS-grad,9.0,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,0
3,53.00000,Private,1,11th,7.0,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,0
4,28.00000,Private,1,Bachelors,13.0,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Other,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,43.00000,Private,1,5th-6th,3.0,Never-married,Machine-op-inspct,Unmarried,White,Female,0,0,40,Other,0
4996,31.00000,Private,1,HS-grad,9.0,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,1
4997,47.00000,Self-emp-inc,1,HS-grad,9.0,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,1
4998,38.58643,Private,1,HS-grad,9.0,Never-married,Adm-clerical,Not-in-family,White,Male,0,0,60,United-States,0


In [ ]:
X3=data3.drop(['income'],axis=1)
X3.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country
0,39.0,State-gov,0,Bachelors,13.0,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States
1,50.0,Self-emp-not-inc,0,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States
2,38.0,Private,1,HS-grad,9.0,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States
3,53.0,Private,1,11th,7.0,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States
4,28.0,Private,1,Bachelors,13.0,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Other


In [ ]:
Y3=data3['income']
Y3.head()

,income
0,0
1,0
2,0
3,0
4,0


Label encoding

In [ ]:
le_x= LabelEncoder()
X3['workclass'] = le_x.fit_transform(X3['workclass'])
X3['education'] = le_x.fit_transform(X3['education'])
X3['marital_status'] = le_x.fit_transform(X3['marital_status'])
X3['occupation'] = le_x.fit_transform(X3['occupation'])
X3['relationship'] = le_x.fit_transform(X3['relationship'])
X3['race'] = le_x.fit_transform(X3['race'])
X3['sex'] = le_x.fit_transform(X3['sex'])


In [ ]:
X3['native_country'] = le_x.fit_transform(X3['native_country'])

In [ ]:
X3

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country
0,39.00000,6,0,10,13.0,4,1,1,4,1,2174,0,40,1
1,50.00000,5,0,10,13.0,2,4,0,4,1,0,0,13,1
2,38.00000,3,1,12,9.0,0,6,1,4,1,0,0,40,1
3,53.00000,3,1,1,7.0,2,6,0,2,1,0,0,40,1
4,28.00000,3,1,10,13.0,2,10,5,2,0,0,0,40,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,43.00000,3,1,4,3.0,4,7,4,4,0,0,0,40,0
4996,31.00000,3,1,12,9.0,2,3,0,4,1,0,0,40,1
4997,47.00000,4,1,12,9.0,2,3,0,4,1,0,0,40,1
4998,38.58643,3,1,12,9.0,4,1,1,4,1,0,0,60,1


Applying logistic

In [ ]:
x3train,x3test,y3train,y3test= train_test_split(X3,Y3,test_size=0.3,random_state=1)

In [ ]:
logmodel = LogisticRegression()
logmodel.fit(x3train,y3train)

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [ ]:
logmodel.score(x3test,y3test)

0.8066666666666666

In [ ]:
predictions = logmodel.predict(x3test)
predictions

array([0, 0, 0, ..., 0, 0, 0])

In [ ]:
print(classification_report(y3test,predictions))

              precision    recall  f1-score   support

           0       0.83      0.95      0.88      1149
           1       0.66      0.35      0.46       351

    accuracy                           0.81      1500
   macro avg       0.75      0.65      0.67      1500
weighted avg       0.79      0.81      0.78      1500



In [ ]:
print(confusion_matrix(y3test, predictions))

[[1086   63]
 [ 227  124]]
